In [1]:
from agent.runtime import AgentConfig
from agent.datasets import ViSTRAdapter

config = AgentConfig.from_yaml("configs/agent/s2_8.yaml")
items = ViSTRAdapter(config.dataset).load(
    split="dev",
    ids=["1"],
)

assert len(items) == 1
item = items[0]

print("ID:", item.id)
print("Video:", item.video_path)
print("Question:", item.question)
print("Options:", item.options)
print("Gold:", item.ground_truth)

ID: 1
Video: /data/ViSTR-Bench-Public/data/Outcome_Prediction/Basketball_Shot/Ego4D/775acd8e-086f-48cf-adf3-c154f0d0bd2d-1.mp4
Question: This is a video of a basketball shot. Please predict whether the basketball will go into the hoop. Answer Yes or No.
Options: ('Yes', 'No')
Gold: No


In [ ]:
import json
import subprocess
import threading
import time
from unittest.mock import patch

import agent.runtime.runner as runner_module
from agent.datasets import ViSTRAdapter
from agent.runtime import AgentConfig, AgentRunner

config = AgentConfig.from_yaml("configs/agent/s2_8.yaml")
items = ViSTRAdapter(config.dataset).load(split="dev", ids=["1"])
assert len(items) == 1, f"Expected one item, got {len(items)}"

item = items[0]
run_id = f"smoke-notebook-{time.strftime('%Y%m%d-%H%M%S')}"
run_dir = config.artifacts.trajectory_root / run_id
print(f"Running ID={item.id}: {item.question}", flush=True)
print("Options:", item.options, flush=True)
print("Run ID:", run_id, flush=True)

# Pi treats a notebook's non-TTY stdin as piped input and waits forever for EOF.
# Close stdin only for subprocesses launched during this rollout.
original_run = subprocess.run

def run_with_closed_stdin(*args, **kwargs):
    kwargs.setdefault("stdin", subprocess.DEVNULL)
    return original_run(*args, **kwargs)

stop_progress = threading.Event()
started = time.monotonic()

def report_progress():
    while not stop_progress.wait(10):
        item_dir = run_dir / item.id
        attempts = sorted(item_dir.glob("attempt-*"))
        sessions = list(item_dir.glob("attempt-*/*.jsonl"))
        attempt = attempts[-1].name if attempts else "initializing"
        print(
            f"[{time.monotonic() - started:.0f}s] rollout active; "
            f"{attempt}; sessions={len(sessions)}",
            flush=True,
        )

progress_thread = threading.Thread(target=report_progress, daemon=True)
progress_thread.start()
try:
    with patch.object(runner_module.subprocess, "run", new=run_with_closed_stdin):
        with AgentRunner(config) as runner:
            records = runner.rollout(
                items,
                skill_content=None,
                run_id=run_id,
            )
finally:
    stop_progress.set()
    progress_thread.join(timeout=1)

record = records[0]
print(json.dumps({
    "id": record.id,
    "predicted_answer": record.predicted_answer,
    "ground_truth": record.extras["ground_truth"],
    "hard": record.hard,
    "agent_ok": record.agent_ok,
    "fail_reason": record.fail_reason,
    "policy_model": record.policy_model,
    "observer_model": record.observer_model,
    "trajectory_dir": record.trajectory_dir,
    "session_html": record.session_html,
    "conversation_path": record.conversation_path,
    "tool_calls": record.n_turns,
    "tool_errors": record.extras["tool_errors"],
}, ensure_ascii=False, indent=2))